In [1]:
import os
from dotenv import load_dotenv
from reanimator.core import Reanimator
from reanimator.labelers import OpenAILabeler, LocalModelLabeler
from reanimator.labelers import TopicChunkPair, calculate_cohens_kappa
load_dotenv()

import nltk
nltk.download('punkt_tab')

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:
reanimator = Reanimator(
    irds_name="irds:cord19/trec-covid",
    email="dummy@gmail.com",
    config={
        "downloader": {
            "email": "dummy@gmail.com"
        }
    }
)

INFO: OpenAILabeler initialized with model: gpt-4.1-mini-2025-04-14


In [3]:
human_judgements = reanimator.source.get_qrels()
topics = reanimator.source.get_topics()
model="qwen/qwen3-30b-a3b"

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.


In [4]:
t1_doc_ids = [judg.doc_id for judg in human_judgements if judg.query_id == "1"]
len(t1_doc_ids)

1647

In [5]:
docs = reanimator.load_documents(doc_ids=t1_doc_ids)
#reanimator.download_documents(docs)
#reanimator.extract_content(docs)
#reanimator.save_documents(docs, "/workspace/data/documents")


Step 1: Loading documents from source...


cord19/trec-covid documents: 100%|██████████| 192509/192509 [00:00<00:00, 194231.77it/s]


In [6]:
chunks = reanimator.chunker.chunk(docs, metadata_fields_to_chunk=["title"])

In [ ]:
#docs = reanimator.load_documents_from_file("/workspace/data/documents")

In [ ]:
#labeler = OpenAILabeler(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
labeler = LocalModelLabeler(model="qwen/qwen3-30b-a3b", 
                            base_url="http://192.168.178.180:1234/v1", 
                            concurrency=10,
                            thinking=False)

In [ ]:
batch = [TopicChunkPair(topic=topics[0], chunk=chunk) for chunk in chunks]

In [ ]:
machine_judgements = await labeler.label_all(batch)

In [ ]:
from reanimator.models import save_judgements
model = model.replace("/", "_")
save_judgements(machine_judgements, f"/workspace/data/judgments/machine_{model}_judgements.json")

In [ ]:
#save_judgements(human_judgements, "/workspace/data/judgments/human_judgements.json")

In [ ]:
calculate_cohens_kappa("/workspace/data/judgments/human_judgements.json", "/workspace/data/judgments/machine_qwen_qwen3-30b-a3b_judgements.json")

In [7]:
from reanimator.retrieval import Indexer, Retriever

indexer = Indexer(index_type="bm25", bm25_path="/workspace/data/indices/bm25_retriever.pkl", max_docs=200)

In [8]:
indexer.index(chunks)

Creating new indexes...
Indexes created and saved.


In [9]:
retriever = Retriever(indexer)

In [10]:
retriever.retrieve(topics[0].query_text)

{'sparse_coronavirus origin': Ranking(query_id='ad-hoc', results=[SearchResult(doc_id='uadfehr6', score=0, rank=0, chunk_id='uadfehr6-metadata-title-0', metadata={}), SearchResult(doc_id='73xil5op', score=0, rank=1, chunk_id='73xil5op-metadata-title-0', metadata={}), SearchResult(doc_id='o877uul1', score=0, rank=2, chunk_id='o877uul1-metadata-title-0', metadata={}), SearchResult(doc_id='w53u5ive', score=0, rank=3, chunk_id='w53u5ive-metadata-title-0', metadata={}), SearchResult(doc_id='es7q6c90', score=0, rank=4, chunk_id='es7q6c90-metadata-title-0', metadata={}), SearchResult(doc_id='hncf2qe8', score=0, rank=5, chunk_id='hncf2qe8-metadata-title-0', metadata={}), SearchResult(doc_id='bgialj4d', score=0, rank=6, chunk_id='bgialj4d-metadata-title-0', metadata={}), SearchResult(doc_id='1ag9jkk6', score=0, rank=7, chunk_id='1ag9jkk6-metadata-title-0', metadata={}), SearchResult(doc_id='beguhous', score=0, rank=8, chunk_id='beguhous-metadata-title-0', metadata={}), SearchResult(doc_id='jkej